In [0]:
book_path = "/Volumes/workspace/default/learning/books.csv"
sale_path = "/Volumes/workspace/default/learning/book_sales.csv"
author_path = "/Volumes/workspace/default/learning/authors.csv"


In [0]:
book = spark.read.csv(book_path, header = True , inferSchema = True)
sale = spark.read.csv(sale_path, header = True , inferSchema = True)
author = spark.read.csv(author_path, header = True , inferSchema = True)
display(book.limit(2))
display(sale.limit(2))
display(author.limit(2))

In [0]:
bsa = (book.join(sale, on="book_id", how="inner").join(author, on="author_id", how="inner"))
display(bsa.limit(2))

In [0]:
ba =  book.join(author, book.author_id == author.author_id, "inner"
).select(
    book.book_id,
    book.title,
    author.name.alias("author_name"),
    book.genre,
    book.price
)
display(ba.limit(2))

In [0]:
from pyspark.sql.functions import col, round
bsa1 = book.join(
    author,
    book.author_id == author.author_id
).join(
    sale,
    book.book_id == sale.book_id
).select(
    book.book_id,
    book.title,
    author.name.alias("author_name"),
    book.genre,
    book.price,
    sale.quantity,
    sale.sale_date,
    round(book.price * sale.quantity, 2).alias("sale_revenue")
    )
bsa1.show()

Which authors generated the most revenue? And what genres are trending?


In [0]:
from pyspark.sql.functions import sum, count, round, col
revenue = bsa1.groupBy("author_name").agg(round(sum(col("sale_revenue")), 2).alias("total_revenue"))
display(revenue)

And what genres are trending?

In [0]:
from pyspark.sql.functions import sum, round

genre = bsa1.groupBy("genre").agg(
    round(sum("quantity"), 0).alias("sold"),
    round(sum("sale_revenue") / sum("quantity"), 2).alias("avg_price")
)
genre.show()
